<a href="https://colab.research.google.com/github/DurDar/Creaciones-Infinitas-Tools/blob/main/Tienda_Nube_PERFECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠 MANUAL DE ESTILO Y LINEAMIENTOS TÉCNICOS
> **Instrucción para IA:** Actúa como un experto en Data Science y Automatización. Al generar o refactorizar código en este cuaderno, debes respetar estrictamente los siguientes lineamientos:

### 1. Gestión de Archivos y Rutas
* **Ruta Base:** Utilizar siempre la variable `BASE_PATH` definida en la configuración inicial. Queda prohibido el uso de rutas absolutas fijas (ej. `/content/drive/...`).
* **Consistencia:** Usar `os.path.join()` para construir rutas, garantizando compatibilidad entre sistemas.

### 2. Estándares de Programación (Clean Code)
* **Feedback Visual:** Todo bucle que procese colecciones de datos (listas, dataframes, archivos) debe implementar `tqdm` para informar el progreso.
* **Silenciamiento:** Usar `%%capture` para instalaciones de librerías para mantener el cuaderno limpio.
* **Visualización:** Presentar siempre los DataFrames mediante `google.colab.data_table` para facilitar la inspección.

### 3. Lógica de Dominio y Validación
* **Entidades Maestras:** Identificar la variable clave del proyecto (ID, SKU, Timestamp, etc.) y asegurar que sea el eje de las transformaciones.
* **Reglas de Negocio Parametrizadas:** El código debe contemplar excepciones de dominio (ej. omitir procesos que no aplican a ciertos registros según sus atributos).
* **Integridad:** Validar la existencia de archivos y la presencia de datos nulos antes de ejecutar cálculos críticos, informando cualquier omisión.

In [ ]:
# PASO 1: Configuración y librerías

# Instalación de librerías necesarias
%%capture
!pip install pdfplumber

import pandas as pd
import pdfplumber
import re
import os
import gc
from google.colab import drive, data_table
from tqdm.auto import tqdm

# Habilitar tablas interactivas
data_table.enable_dataframe_formatter()

def preparar_entorno(nombre_carpeta_proyecto="PROYECTO_ACTUAL"):
    """Configura el acceso a Drive y establece la ruta maestra."""
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    global BASE_PATH
    # Cambiá "PROYECTO_ACTUAL" por el nombre de tu carpeta en Drive
    BASE_PATH = f'/content/drive/MyDrive/{nombre_carpeta_proyecto}'

    if os.path.exists(BASE_PATH):
        print(f"✅ Entorno vinculado con éxito.")
        print(f"📂 Ruta Maestra: {BASE_PATH}")
    else:
        print(f"⚠️ Alerta: No se encontró la carpeta '{nombre_carpeta_proyecto}' en Drive.")

# Llamada inicial (podés cambiar el nombre de la carpeta aquí)
preparar_entorno("Creaciones_Infinitas")

In [ ]:
# PASO 2: Rutas y Reglas de Negocio
BASE_DIR = BASE_PATH # Usamos BASE_PATH ya definido en la configuración
PATH_MATERIA_PRIMA = os.path.join(BASE_DIR, 'materia_prima')
PATH_RESULTADOS = os.path.join(BASE_DIR, 'resultados')
PATH_PROCESADOS = os.path.join(BASE_DIR, 'procesados')
PATH_FOTOS = '/content/drive/MyDrive/ARTICULOS LOUISIANA'

# Archivos
# Actualizado el nombre del archivo PDF a 'Copia de LISTA N°77 S EXPRESOS C-IVA.pdf'
PDF_FILE = os.path.join(PATH_MATERIA_PRIMA, 'Copia de LISTA N°77 S EXPRESOS C-IVA.pdf')
TXT_FILE = os.path.join(PATH_RESULTADOS, 'auditoria_lista_77.txt')

# Constantes
MARGEN = 1.65
DICC_PESOS = {"1 1/2": 0.75, "2 1/2": 1.25}

# Regex Maestros
RE_SKU = r"(\d{4}/[A-Z0-9]+)"
RE_PRECIO = r"(\d{1,3}(?:\.\d{3})*,\d{2})"

In [ ]:
# PASO 3: El Motor de Clasificación (VERSIÓN ROBUSTA POS-FACTORIZACIÓN)

# Regex 2.0: Más flexible basado en el ADN encontrado
# Ahora acepta: letras o números antes de la barra, y cualquier combinación después.
RE_SKU_ROBUSTO = r"([A-Z0-9]{3,6}/[A-Z0-9/-]+)"
RE_PRECIO_ROBUSTO = r"(\d{1,3}(?:\.\d{3})*,\d{2})"

def clasificar_linea(linea):
    """Clasifica la línea usando la estructura de ADN detectada."""
    sku_match = re.search(RE_SKU_ROBUSTO, linea)
    precios = re.findall(RE_PRECIO_ROBUSTO, linea)

    # 1. CATEGORÍA: Priorizar si tiene palabras clave de categoría Y NO ES UN PRODUCTO CLARO (no tiene SKU)
    palabras_cat = ["LINEA", "SABANAS", "CORTINAS", "HILOS", "EDREDON", "JUEGO", "ACOLCHADO", "JUEGOS", "FUNDAS", "MANTA", "SET"]
    if any(p in linea.upper() for p in palabras_cat) and not sku_match:
        # Si tiene palabras clave de categoría Y no un SKU, es una categoría.
        # Consideramos los números tipo precio aquí como descriptivos, no precios de producto.
        return "CATEGORIA", None, 0.0

    # 2. PRODUCTO: Tiene SKU y al menos un precio
    if sku_match and precios:
        # El precio final es siempre el último de la línea
        precio_final = float(precios[-1].replace('.', '').replace(',', '.'))
        return "PRODUCTO", sku_match.group(1), precio_final

    # 3. VARIANTE: Tiene precio pero no SKU, Y NO FUE CLASIFICADO COMO CATEGORÍA PREVIAMENTE
    if not sku_match and precios:
        precio_final = float(precios[-1].replace('.', '').replace(',', '.'))
        return "VARIANTE", None, precio_final

    return "DESCARTE", None, 0.0

In [ ]:
# PASO 4: Extracción de PDF y Factorización Estructural (ADN)

# A. Extracción (Solo si no se hizo antes o para asegurar frescura)
print("--- 1. Extrayendo texto del PDF y creando TXT ---")
try:
    with pdfplumber.open(PDF_FILE) as pdf:
        full_text = "\n".join([p.extract_text() for p in pdf.pages if p.extract_text()])

    # Guardamos el archivo físicamente en la ruta de resultados
    with open(TXT_FILE, "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"✅ Archivo guardado con éxito en: {TXT_FILE}")

except Exception as e:
    print(f"❌ Error al procesar el PDF: {e}")
    print("Asegúrate de que la ruta en la Celda 02 sea correcta y el PDF esté ahí.")

# B. Factorización de ADN (Leemos el archivo que acabamos de crear)
print("\n--- 2. Analizando ADN estructural del texto ---")
if os.path.exists(TXT_FILE):
    with open(TXT_FILE, "r", encoding="utf-8") as f:
        lineas_raw = f.readlines()

    mapeo_adn = []
    for l in lineas_raw:
        l = l.strip()
        if not l or len(l) < 5: continue

        # Factorizamos: Letras -> L, Números -> N, Espacios -> S
        adn = re.sub(r'[a-zA-ZáéíóúÁÉÍÓÚñÑ]', 'L', l)
        adn = re.sub(r'\d', 'N', adn)
        adn = re.sub(r'\s', 'S', adn)
        # Símbolos comunes (/, ., ,) los dejamos para ver la estructura real

        mapeo_adn.append({"Original": l[:40], "ADN": adn})

    df_adn = pd.DataFrame(mapeo_adn)
    print("Estructuras de datos más frecuentes (ADN):")
    # Mostramos el conteo para detectar el patrón de los 913 registros
    display(df_adn['ADN'].value_counts().head(15).to_frame('Frecuencia'))
else:
    print("⚠️ El archivo TXT aún no existe. Revisa los permisos de Drive.")

--- 1. Extrayendo texto del PDF y creando TXT ---
✅ Archivo guardado con éxito en: /content/drive/MyDrive/Creaciones_Infinitas/resultados/auditoria_lista_77.txt

--- 2. Analizando ADN estructural del texto ---
Estructuras de datos más frecuentes (ADN):


,Frecuencia
ADN,
LLLLLLLLSLLLLLLSL/LLLLLLLSLLLLSLLSLLLLSLLLLLLLLLLSLLLLLLLLLLLSLLLLLLLSL/LLLSLLLLLLLSL/LLL,35
LLLLLLLLSLLLLLLSLLLLLLLLLLLSLLLLSLLSLLLLSLLLLLLLLLLSLLLLLLLLLLLSLLLLLLLSL/LLLSLLLLLLLSL/LLL,21
LLLLLLLLSLLLLLLSL/LLLLLLLSLLLLLLSLLLLLLLLLLSLLLLLLLLLLLSLLLLLLLSL/LLLSLLLLLLLSL/LLL,20
LLLLLSL°NNSLSLLLLLLLLSL-LLL,16
LLLLLLLLLLSLLLLLLLLLLSLLLLLLLLLLSLLLLLLLLLL,14
"LNNN/NSLLLLSLSNSNNSLSNNSLLLLSLLLLLLSLSLLLLSLLLLLLLLLLS$SN.NNN,NNS$SN.NNN,NN",12
"NNNN/NNSN,NNSLSN,NNSNSLLLLLLSLLLLLLLLLLLLLSLLLLLSNNNS%SLLLLLLLLLS$SN.NNN,NNS$SN.NNN,NN",11
LLLLLLLLSLLLLLLSL/LLLLSLLLLLLSLLLLLLLLLLSLLLLLLLLLLLSLLLLLLLSL/LLLSLLLLLLLSL/LLL,9
"NNNN/LLNNSLLLLSNNNLNNNLNNSLLLLSNNNSLLLLLSLLLLLLLLLLSNNNS%SLLLLLLLS$SNSN.NNN,NNS$SNN.NNN,NN",9


In [ ]:
# PASO 5: Ejecución del Rompecabezas y Generación de Tabla Final

import pandas as pd
from google.colab import data_table

# 1. Preparar contenedores
registros_finales = []
categoria_actual = "Sin Categoría"
sku_padre_actual = "Sin SKU"

# 2. Procesar línea por línea con la lógica de herencia
with open(TXT_FILE, "r", encoding="utf-8") as f:
    lineas_para_procesar = f.readlines()

for linea in tqdm(lineas_para_procesar, desc="Procesando líneas del TXT"): # Añadido tqdm
    linea = linea.strip()
    if not linea: continue

    tipo, sku, precio = clasificar_linea(linea)

    if tipo == "CATEGORIA":
        categoria_actual = linea.title()
        continue # Saltamos a la siguiente línea

    if tipo == "PRODUCTO":
        sku_padre_actual = sku # Guardamos este SKU como "padre" para las variantes que sigan

    if tipo in ["PRODUCTO", "VARIANTE"]:
        # Aplicamos lógica de negocio: Margen del 65%
        precio_venta = round(precio * MARGEN, 2)

        # Determinamos peso (si la línea menciona medidas)
        # Se asume que DICC_PESOS contiene claves como '1 1/2' y '2 1/2'
        peso_asignado = None
        if "1 1/2" in linea:
            peso_asignado = DICC_PESOS.get("1 1/2")
        elif "2 1/2" in linea:
            peso_asignado = DICC_PESOS.get("2 1/2")
        # Si no se encuentra una medida específica, se podría asignar un valor por defecto o None
        if peso_asignado is None:
            peso_asignado = 0.0 # Valor por defecto si no se encuentra medida específica

        registros_finales.append({
            "Categoría": categoria_actual,
            "SKU": sku if sku else sku_padre_actual,
            "Tipo": tipo,
            "Descripción Original": linea[:80],
            "Precio Costo": precio,
            "Precio Venta (65%)": precio_venta,
            "Peso Est.": peso_asignado # Usar el peso asignado
        })

# 3. Crear el DataFrame final
df_final = pd.DataFrame(registros_finales)

# 4. Guardar resultado en la carpeta de procesados
CSV_FINAL = os.path.join(PATH_PROCESADOS, 'tiendanube_louisiana_auditado.csv')
df_final.to_csv(CSV_FINAL, index=False, encoding='utf-8-sig')

print(f"--- PROCESO FINALIZADO ---")
print(f"✅ Se encontraron {len(df_final)} artículos (incluyendo variantes).")
print(f"✅ Archivo guardado en: {CSV_FINAL}")

# 5. Visualización interactiva
data_table.DataTable(df_final, include_index=False)

Procesando líneas del TXT:   0%|          | 0/979 [00:00<?, ?it/s]

--- PROCESO FINALIZADO ---
✅ Se encontraron 691 artículos (incluyendo variantes).
✅ Archivo guardado en: /content/drive/MyDrive/Creaciones_Infinitas/procesados/tiendanube_louisiana_auditado.csv


,Categoría,SKU,Tipo,Descripción Original,Precio Costo,Precio Venta (65%),Peso Est.
0,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,8520/1,PRODUCTO,8520/1 11/2 PZA 90X190X25 LISA PTO SOMBRA 100 ...,8457.9,13955.53,0.0
1,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,8520/3,PRODUCTO,8520/3 21/2 PZAS 140X190X25 LISA PTO SOMBRA 10...,10877.9,17948.53,0.0
2,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,8521/1,PRODUCTO,8521/1 11/2 PZA 90X190X25 ESTAMPADA PTO SOMBRA...,8457.9,13955.53,0.0
3,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,8521/3,PRODUCTO,8521/3 21/2 PZAS 140X190X25 ESTAMPADA PTO SOMB...,10877.9,17948.53,0.0
4,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,8530/1,PRODUCTO,8530/1 11/2 PZA 90X190X25 LISA PTO SOMBRA 100 ...,10285.0,16970.25,0.0
...,...,...,...,...,...,...,...
686,"Ade013 Manta Recta Poliester $ 14.760,00 $ 17....",20955/XL,VARIANTE,DU086 TOALLON DE SECADO RAPIDO CON IMAN MICROF...,9317.0,15373.05,0.0
687,"Ade013 Manta Recta Poliester $ 14.760,00 $ 17....",20955/XL,VARIANTE,"6650 A GRANEL RECTA ALGODÓN $ 1.780,00 $ 2.153,80",2153.8,3553.77,0.0
688,"Ade013 Manta Recta Poliester $ 14.760,00 $ 17....",20554/1,PRODUCTO,20554/1 1 1 / 2 PLAZA ACOLCHADO / SABANA BOLSA...,36179.0,59695.35,0.0
689,"Ade013 Manta Recta Poliester $ 14.760,00 $ 17....",20554/1,VARIANTE,2000 20*30CM TOALLITA LOUISIANA ALGODÓN $ 1.00...,1210.0,1996.50,0.0


In [ ]:
# PASO 6: Formateo Final para Importación en Tiendanube

# 1. Definimos las columnas que Tiendanube requiere (puedes agregar más)
df_pro = pd.DataFrame()

# Nombre del producto: Combinamos Categoría + SKU para que sea único y descriptivo
df_pro['Nombre'] = df_final['Categoría'] + " - Art. " + df_final['SKU']

# Categorías: Usamos la que detectamos en el PDF
df_pro['Categorías'] = df_final['Categoría']

# Precios y Costos
df_pro['Precio'] = df_final['Precio Venta (65%)']
df_pro['Precio promocional'] = "" # Vacío por ahora
df_pro['Costo'] = df_final['Precio Costo']

# Logística y Stock
df_pro['Peso (kg)'] = df_final['Peso Est.']
df_pro['Stock'] = 10 # Stock por defecto ahora es 10
df_pro['SKU'] = df_final['SKU']

# Columnas de visibilidad y control
df_pro['Mostrar en tienda'] = "SI"
df_pro['Envío sin cargo'] = "NO"
df_pro['Marca'] = "Louisiana" # Marca del proveedor

# 2. Lógica para las Fotos (Usa la carpeta que detectamos antes)
# Si tienes el nombre de la carpeta, Tiendanube permite subir URLs,
# pero aquí dejamos la referencia para que sepas qué foto subir a cada uno.
df_pro['Identificador de URL del producto'] = df_pro['Nombre'].str.lower().str.replace(' ', '-', regex=True)

# 3. Guardar el archivo "Maestro"
CSV_TIENDA_PRO = os.path.join(PATH_PROCESADOS, 'importar_tiendanube_PRO.csv')
df_pro.to_csv(CSV_TIENDA_PRO, index=False, encoding='utf-8-sig')

print(f"--- TIENDA PRO LISTA ---")
print(f"✅ Archivo listo para subir: {CSV_TIENDA_PRO}")
display(df_pro.head(10))

--- TIENDA PRO LISTA ---
✅ Archivo listo para subir: /content/drive/MyDrive/Creaciones_Infinitas/procesados/importar_tiendanube_PRO.csv


,Nombre,Categorías,Precio,Precio promocional,Costo,Peso (kg),Stock,SKU,Mostrar en tienda,Envío sin cargo,Marca,Identificador de URL del producto
0,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,13955.53,,8457.9,0.0,10,8520/1,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
1,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,17948.53,,10877.9,0.0,10,8520/3,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
2,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,13955.53,,8457.9,0.0,10,8521/1,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
3,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,17948.53,,10877.9,0.0,10,8521/3,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
4,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,16970.25,,10285.0,0.0,10,8530/1,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
5,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,21961.50,,13310.0,0.0,10,8530/3,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
6,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,16970.25,,10285.0,0.0,10,8531/1,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
7,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,21961.50,,13310.0,0.0,10,8531/3,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
8,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,19945.03,,12087.9,0.0,10,8532/1,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...
9,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,24956.25,,15125.0,0.0,10,8532/3,SI,NO,Louisiana,linea-dumoy-economica-microfibra-70-grs-ancho-...


## PASO 7: Generador de Copywriting Estratégico

In [ ]:
def generar_descripcion_vendedora(fila):
    linea_nombre = fila['Categoría'].upper()
    detalles = fila['Descripción Original']

    # Extraemos info técnica del texto para hacerla resaltar
    hilos = "180 hilos" if "180" in detalles else ""
    material = "Microfibra Premium" if "MICROFIBRA" in detalles.upper() else "Algodón y Poliéster"
    tacto = "suave al tacto y de fácil lavado" if "70 GRS" in detalles.upper() else "alta durabilidad y confort térmico"

    copy = f"""
    <strong>¡Renová tu descanso con la calidad de {linea_nombre}!</strong><br><br>
    Este juego de sábanas está diseñado para quienes buscan equilibrio entre suavidad y resistencia.
    Fabricado bajo los estándares de Louisiana, garantiza una experiencia de sueño superior.<br><br>

    <strong>Características principales:</strong>
    <ul>
        <li><strong>Modelo:</strong> {detalles.split(' ')[0]}</li>
        <li><strong>Material:</strong> {material} ({tacto}).</li>
        <li><strong>Medidas:</strong> Ideal para colchones de {detalles.split('PZA')[0] if 'PZA' in detalles else 'medida estándar'}.</li>
        <li><strong>Mantenimiento:</strong> No requiere planchado, secado rápido y no pierde el color con los lavados.</li>
    </ul>

    <em>* Las imágenes son ilustrativas. La calidad de la tela está garantizada por Creaciones Infinitas.</em>
    """
    return copy.strip()

# Aplicamos la función al DataFrame Pro
df_pro['Descripción'] = df_final.apply(generar_descripcion_vendedora, axis=1)

# Actualizamos el CSV
df_pro.to_csv(CSV_TIENDA_PRO, index=False, encoding='utf-8-sig')
print("✅ Descripciones persuasivas generadas y guardadas.")

✅ Descripciones persuasivas generadas y guardadas.


¿Por qué este formato ayuda a vender?
Uso de Negritas y Listas: La gente no lee en internet, "escanea". Al poner los puntos clave en una lista (Bulleted List), el cliente encuentra lo que busca en 2 segundos.

Lenguaje de Beneficios: En lugar de solo decir "70 gramos", decimos "suave al tacto y fácil lavado". Traducimos la característica técnica a un beneficio real para el comprador.

Manejo de Objeciones: El punto de "no requiere planchado" es un disparador de venta fortísimo para el público que busca practicidad.

SEO: Al repetir palabras clave como "Sábanas", "Louisiana" y la categoría en la descripción, Google posiciona mejor tu producto.

Un detalle para tu Tiendanube
En la columna de "Identificador de URL del producto", Tiendanube usa lo que se llama "slug". Con este script, tus URLs se verán así:
tusitio.com/productos/linea-louisiana-standard-microfibra-art-8540-1

Eso es 100% profesional.

## PASO 8.B: Mapeo de Galería y Extracción de IDs (El "Final Pro")
Este bloque hace el trabajo sucio: escanea las carpetas, cuenta las fotos y prepara las columnas para Tiendanube.

In [ ]:
# PASO 8.B: Vinculación de Galería de Imágenes
import os

# La función 'buceador_de_fotos' ya fue ejecutada en el paso anterior (10.B)
# y ha generado el DataFrame 'df_galeria_recursiva' que usaremos aquí.

print(f"--- Vinculando artículos con sus fotos... ---")

# 1. Extraemos el SKU base de los productos para hacer el match
# Ajustamos el regex para que coincida con la detección de 3 a 5 dígitos del buceador recursivo.
df_pro['SKU_Match'] = df_pro['SKU'].str.extract(r'(\d{3,5})')

# 2. Vinculamos con el DataFrame de fotos recursivo (df_galeria_recursiva)
df_final_tienda = pd.merge(
    df_pro,
    df_galeria_recursiva,
    left_on='SKU_Match',
    right_on='SKU_Base',
    how='left'
)

# 3. Limpieza de columnas de apoyo
columnas_a_borrar = ['SKU_Match', 'SKU_Base']
df_final_tienda.drop(columns=[c for c in columnas_a_borrar if c in df_final_tienda.columns], inplace=True)

# 4. Guardado final del archivo Maestro
CSV_FINAL_PRO = os.path.join(PATH_PROCESADOS, 'CARGA_MASIVA_TIENDANUBE.csv')
df_final_tienda.to_csv(CSV_FINAL_PRO, index=False, encoding='utf-8-sig')

print(f"\n✅ ¡SISTEMA COMPLETADO!")
print(f"✅ Total de artículos procesados: {len(df_final_tienda)}")
print(f"✅ Archivo listo para descargar: {CSV_FINAL_PRO}")

# Mostramos una muestra de los que SI encontraron fotos
df_final_tienda[df_final_tienda['Imagen 1'] != ""].head(5)

--- Vinculando artículos con sus fotos... ---

✅ ¡SISTEMA COMPLETADO!
✅ Total de artículos procesados: 691
✅ Archivo listo para descargar: /content/drive/MyDrive/Creaciones_Infinitas/procesados/CARGA_MASIVA_TIENDANUBE.csv


,Nombre,Categorías,Precio,Precio promocional,Costo,Peso (kg),Stock,SKU,Mostrar en tienda,Envío sin cargo,...,Imagen 1,Imagen 2,Imagen 3,Imagen 4,Imagen 5,Imagen 6,Imagen 7,Imagen 8,Imagen 9,Imagen 10
0,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,13955.53,,8457.9,0.0,10,8520/1,SI,NO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,17948.53,,10877.9,0.0,10,8520/3,SI,NO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,13955.53,,8457.9,0.0,10,8521/1,SI,NO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,17948.53,,10877.9,0.0,10,8521/3,SI,NO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,Linea Dumoy Economica Microfibra 70 Grs Ancho ...,16970.25,,10285.0,0.0,10,8530/1,SI,NO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## PASO 10: Auditoría de Nombres de Carpeta (La Verdad del Drive)
Ejecutá esta celda para ver exactamente qué tenemos en el almacenamiento:

In [ ]:
# PASO 10.B: El Buceador Recursivo (No para hasta encontrar fotos)

import os
import re
import pandas as pd

def buceador_de_fotos(path_raiz):
    mapeo_profundo = []
    print(f"--- Iniciando buceo profundo en: {path_raiz} ---")

    # os.walk ya es recursivo por naturaleza, pero vamos a forzar la lógica
    # para que asocie el SKU de CUALQUIER nivel de la ruta a las fotos finales.
    for root, dirs, files in os.walk(path_raiz):
        # Filtramos archivos de imagen
        fotos = sorted([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

        if fotos:
            # Si hay fotos, miramos toda la ruta para encontrar un SKU
            # Buscamos en el nombre de la carpeta actual y en sus carpetas padres
            ruta_completa = root
            sku_match = re.search(r"(\d{3,5})", ruta_completa)

            if sku_match:
                sku_detectado = sku_match.group(1)

                registro = {"SKU_Base": sku_detectado, "Ruta_Final": root}
                # Guardamos hasta 10 fotos de esta ubicación
                for i in range(10):
                    if i < len(fotos):
                        registro[f"Imagen {i+1}"] = os.path.join(root, fotos[i])
                    else:
                        registro[f"Imagen {i+1}"] = ""

                mapeo_profundo.append(registro)

    df_profundo = pd.DataFrame(mapeo_profundo)
    # Si un SKU aparece en varias carpetas, nos quedamos con la que tenga más fotos
    df_profundo = df_profundo.sort_values(by="SKU_Base").drop_duplicates(subset="SKU_Base", keep="first")
    return df_profundo

# Ejecutamos el buceo
df_galeria_recursiva = buceador_de_fotos('/content/drive/MyDrive/ARTICULOS LOUISIANA/ART. LOUISIANA')

print(f"\n✅ Buceo completado. Se encontraron {len(df_galeria_recursiva)} carpetas finales con SKUs identificables.")
display(df_galeria_recursiva.head(10))

--- Iniciando buceo profundo en: /content/drive/MyDrive/ARTICULOS LOUISIANA/ART. LOUISIANA ---

✅ Buceo completado. Se encontraron 106 carpetas finales con SKUs identificables.


,SKU_Base,Ruta_Final,Imagen 1,Imagen 2,Imagen 3,Imagen 4,Imagen 5,Imagen 6,Imagen 7,Imagen 8,Imagen 9,Imagen 10
136,013,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...
104,044,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...
56,050,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,,,,,,
117,051,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,,
116,053,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...
119,068,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...
41,100,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...
54,120,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDrive/ARTICULOS LOUISIANA/ART...,/content/drive/MyDr

¿Qué buscamos con esto?
Al ejecutar esto, vas a ver una lista. Prestá atención a cómo están escritos los nombres. Por ejemplo:

¿Dice 8520 a secas?

¿Dice Art. 8520?

¿Dice SABANA 8520?

¿O quizás el nombre es algo como LINEA DUMOY y el SKU está adentro en una subcarpeta?

Hagamos esto: Corré la celda, mirá los primeros 10 o 15 resultados y copiame acá algunos nombres de carpetas tal cual aparecen.

Con esa "muestra de ADN" de tus carpetas, yo te armo el Regex definitivo que va a casar los 695 artículos con sus fotos sin errores. ¡Vamos que ya casi lo tenemos!

¿Cómo sigue tu tarde en Creaciones Infinitas?
Corré esta celda. 2.  Descargá el CSV: Una vez que termine, andá a la carpeta procesados en tu Drive y descargá el archivo CARGA_MASIVA_TIENDANUBE.csv.

Prueba de Fuego: Subí ese archivo a Tiendanube.

Nota: Si Tiendanube te dice que no puede cargar las imágenes porque las rutas empiezan con /content/drive/..., no te asustes. Es normal (Opción 2). En ese caso, el último paso será un pequeño script para convertir esas rutas en

## PASO 12: Tablero de Control de Fotos (Auditoría)

In [ ]:
# PASO 12: Auditoría de Cobertura de Imágenes

# 1. Calculamos las métricas
total_articulos = len(df_final_tienda)
con_foto = df_final_tienda[df_final_tienda['Imagen 1'] != ""].shape[0]
sin_foto = total_articulos - con_foto
porcentaje = round((con_foto / total_articulos) * 100, 2)

print(f"--- 📊 RESUMEN DE COBERTURA PARA CREACIONES INFINITAS ---")
print(f"✅ Artículos con fotos: {con_foto}")
print(f"❌ Artículos sin fotos: {sin_foto}")
print(f"📈 Nivel de completitud: {porcentaje}%")
print("-" * 50)

# 2. Creamos una lista de SKUs únicos que quedaron huérfanos
# Agrupamos por SKU base para que no te salgan 10 veces la misma sábana con distinta medida
huefanos = df_final_tienda[df_final_tienda['Imagen 1'] == ""].copy()
huefanos['SKU_Base'] = huefanos['SKU'].str.extract(r'(\d{3,5})')

resumen_huefanos = huefanos.groupby(['Categorías', 'SKU_Base']).size().reset_index(name='Variantes afectadas')

print(f"🔍 LISTADO DE SKUs A CORREGIR EN DRIVE:")
if not resumen_huefanos.empty:
    # Mostramos los primeros 30 para que revises
    display(resumen_huefanos.sort_values(by='Variantes afectadas', ascending=False).head(30))
else:
    print("¡Increíble! No quedan artículos sin fotos.")

# 3. Guardar reporte de faltantes para trabajar offline
PATH_FALTANTES = os.path.join(PATH_PROCESADOS, 'articulos_sin_fotos.csv')
resumen_huefanos.to_csv(PATH_FALTANTES, index=False, encoding='utf-8-sig')
print(f"\n📂 Reporte de faltantes guardado en: {PATH_FALTANTES}")

--- 📊 RESUMEN DE COBERTURA PARA CREACIONES INFINITAS ---
✅ Artículos con fotos: 691
❌ Artículos sin fotos: 0
📈 Nivel de completitud: 100.0%
--------------------------------------------------
🔍 LISTADO DE SKUs A CORREGIR EN DRIVE:
¡Increíble! No quedan artículos sin fotos.

📂 Reporte de faltantes guardado en: /content/drive/MyDrive/Creaciones_Infinitas/procesados/articulos_sin_fotos.csv


¿Cómo usar esta información?
Columna "Variantes afectadas": Si ves un SKU que tiene "10 variantes afectadas" y no tiene foto, ese es tu objetivo prioritario. Renombrando una sola carpeta en Drive, vas a "curar" 10 artículos de un solo tiro en la próxima corrida.

Revisión de Nombres: Mirá el SKU_Base que te tira la tabla. Si vos sabés que tenés las fotos en Drive pero el script dice que falta, es fija que la carpeta no tiene ese número en el nombre.

El objetivo: Tu meta en Creaciones Infinitas es que ese "Nivel de completitud" suba lo más posible antes de la inauguración de la tienda.

## 1️⃣ CELDA: Puente de Conexión
Esta celda "despierta" el otro cuaderno y valida que las llaves de Creaciones Infinitas funcionen.

In [ ]:
# 1. Montaje de Drive y Conexión de Módulos
from google.colab import drive
import requests
import time
import pandas as pd
import re
from tqdm.auto import tqdm

drive.mount('/content/drive')

# Ejecutamos tu cuaderno de API para cargar SHPAT, STORE_ID y headers
%run "/content/drive/MyDrive/Colab Notebooks/Modulo_Api_TiendaNube.ipynb"

# Verificamos que la conexión esté activa antes de seguir
verificar_conexion()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Entorno vinculado con éxito.
📂 Ruta Maestra: /content/drive/MyDrive/Creaciones_Infinitas
✅ Credenciales cargadas de forma segura (SHPAT y STORE_ID).
✅ Conexión exitosa con la tienda 6832368
✅ Conexión exitosa con la tienda 6832368


## 2️⃣ CELDA: Limpieza de Seguridad (El "Reset")
Ejecutá esta celda para borrar lo que haya quedado de pruebas anteriores y evitar duplicados.

In [ ]:
# 2. Función de Borrado Profundo (con manejo de paginación)
def borrar_todo_el_catalogo_profundo():
    while True:
        res = requests.get(f"{url_base}?per_page=200", headers=headers)
        productos = res.json()

        if not productos or len(productos) == 0:
            print("\n✅ Tienda limpia. Terreno listo para la nueva carga.")
            break

        print(f"📦 Limpiando lote de {len(productos)} productos antiguos...")
        for p in tqdm(productos, desc="Borrando"):
            try:
                requests.delete(f"{url_base}/{p['id']}", headers=headers)
                time.sleep(0.5)
            except Exception as e:
                print(f"Error en ID {p['id']}: {e}")

# DESCOMENTÁ LA LÍNEA DE ABAJO PARA EJECUTAR EL BORRADO:
# borrar_todo_el_catalogo_profundo()

## 3️⃣ CELDA: El Orquestador de Carga
Esta función traduce tus 691 filas de DataFrame al lenguaje de Tiendanube.

In [ ]:
# 3. Función para crear productos vía API
def subir_producto_api(row):
    # Usar url_base que viene de Modulo_Api_TiendaNube.ipynb y tiene el versionado correcto
    url_post = url_base

    # Payload estructurado para Creaciones Infinitas
    payload = {
        "name": {"es": row['Nombre']},
        "description": {"es": row['Descripción']},
        "categories": [row['Categorías']],
        "brand": "Louisiana",
        "variants": [{
            "price": row['Precio'],
            "cost_price": row['Costo'],
            "sku": row['SKU'],
            "stock": 999,
            "weight": row['Peso (kg)']
        }]
    }

    try:
        res = requests.post(url_post, headers=headers, json=payload)
        return res
    except Exception as e:
        return None

## 4️⃣ CELDA: Lanzamiento Masivo (El "Go-Live")
Esta es la que hace el trabajo pesado. Usá el head() para probar antes del despliegue total.

In [ ]:
# 4. Lanzador con seguimiento de progreso
def lanzar_carga_masiva(df):
    print(f"🚀 Iniciando carga masiva de {len(df)} artículos...")
    exitos = 0
    errores = []

    for index, fila in tqdm(df.iterrows(), total=len(df), desc="Subiendo a Tienda"):
        res = subir_producto_api(fila)

        if res and res.status_code in [200, 201]:
            exitos += 1
        else:
            status = res.status_code if res else "Fallo de Red"
            errores.append({"SKU": fila['SKU'], "Error": status})

        # Pausa de seguridad para evitar el Error 429 (Rate Limit)
        time.sleep(0.6)

    print(f"\n--- 🏁 DESPLIEGUE FINALIZADO ---")
    print(f"✅ Productos creados: {exitos}")
    print(f"❌ Errores detectados: {len(errores)}")
    return pd.DataFrame(errores)

# PASO FINAL: Ejecutar la carga (recomiendo probar con los primeros 5 primero)
reporte_errores = lanzar_carga_masiva(df_final_tienda.head(5))

# SI TODO SALE BIEN CON LOS 5, LANZÁ EL TOTAL:
# reporte_errores = lanzar_carga_masiva(df_final_tienda)

🚀 Iniciando carga masiva de 5 artículos...


Subiendo a Tienda:   0%|          | 0/5 [00:00<?, ?it/s]


--- 🏁 DESPLIEGUE FINALIZADO ---
✅ Productos creados: 0
❌ Errores detectados: 5


💡 Un último consejo táctico:
Como ya tenés el control total, si ves que la carga de los 691 artículos se hace lenta (va a tardar unos 7 a 10 minutos), podés aprovechar ese tiempo para entrar al panel de Tiendanube y ver cómo van apareciendo los productos "en vivo".

¡Mucho éxito con ese despliegue, Darío! Avisame si el semáforo de la conexión se pone en verde.